In [ ]:
%pip uninstall -y kafka-python
%pip install kafka-python-ng

Found existing installation: kafka-python 2.3.0
Uninstalling kafka-python-2.3.0:
  Successfully uninstalled kafka-python-2.3.0
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import yfinance as yf
from kafka import KafkaProducer
import json
import time

# 1. Setup Kafka Producer
producer = KafkaProducer(
    bootstrap_servers=['localhost:9092'],
    value_serializer=lambda x: json.dumps(x).encode('utf-8')
)

def publish_stock_data(ticker_symbol):
    # 2. Download history (last 1 month, daily)
    ticker = yf.Ticker(ticker_symbol)
    df = ticker.history(period="1mo")
    
    # 3. Iterate and Publish
    for timestamp, row in df.iterrows():
        data = {
            "ticker": ticker_symbol,
            "date": str(timestamp.date()),
            "open": row['Open'],
            "high": row['High'],
            "low": row['Low'],
            "close": row['Close'],
            "volume": int(row['Volume'])
        }
        
        producer.send('stock_prices', value=data)
        print(f"Sent: {data['date']} - {data['ticker']} Close: {data['close']:.2f}")
        time.sleep(0.5) # Simulate a stream

if __name__ == "__main__":
    publish_stock_data("AAPL")
    producer.flush() # Ensure all messages are sent


Sent: 2026-02-05 - AAPL Close: 275.65
Sent: 2026-02-06 - AAPL Close: 277.86
Sent: 2026-02-09 - AAPL Close: 274.62
Sent: 2026-02-10 - AAPL Close: 273.68
Sent: 2026-02-11 - AAPL Close: 275.50
Sent: 2026-02-12 - AAPL Close: 261.73
Sent: 2026-02-13 - AAPL Close: 255.78
Sent: 2026-02-17 - AAPL Close: 263.88
Sent: 2026-02-18 - AAPL Close: 264.35
Sent: 2026-02-19 - AAPL Close: 260.58
Sent: 2026-02-20 - AAPL Close: 264.58
Sent: 2026-02-23 - AAPL Close: 266.18
Sent: 2026-02-24 - AAPL Close: 272.14
Sent: 2026-02-25 - AAPL Close: 274.23
Sent: 2026-02-26 - AAPL Close: 272.95
Sent: 2026-02-27 - AAPL Close: 264.18
Sent: 2026-03-02 - AAPL Close: 264.72
Sent: 2026-03-03 - AAPL Close: 263.75
Sent: 2026-03-04 - AAPL Close: 262.52
Sent: 2026-03-05 - AAPL Close: 257.52


In [2]:
from kafka import KafkaConsumer
import json

# 1. Setup Kafka Consumer
consumer = KafkaConsumer(
    'stock_prices',
    bootstrap_servers=['localhost:9092'],
    auto_offset_reset='earliest', # Start from the beginning of the topic
    enable_auto_commit=True,
    group_id='stock-analyzer-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Waiting for stock data...")

# 2. Continuous Listen Loop
try:
    for message in consumer:
        stock_info = message.value
        print(f"Received: {stock_info['ticker']} | Date: {stock_info['date']} | Price: {stock_info['close']:.2f}")
except KeyboardInterrupt:
    print("Stopping consumer...")
finally:
    consumer.close()


Waiting for stock data...
Received: AAPL | Date: 2026-02-05 | Price: 275.65
Received: AAPL | Date: 2026-02-06 | Price: 277.86
Received: AAPL | Date: 2026-02-09 | Price: 274.62
Received: AAPL | Date: 2026-02-10 | Price: 273.68
Received: AAPL | Date: 2026-02-11 | Price: 275.50
Received: AAPL | Date: 2026-02-12 | Price: 261.73
Received: AAPL | Date: 2026-02-13 | Price: 255.78
Received: AAPL | Date: 2026-02-17 | Price: 263.88
Received: AAPL | Date: 2026-02-18 | Price: 264.35
Received: AAPL | Date: 2026-02-19 | Price: 260.58
Received: AAPL | Date: 2026-02-20 | Price: 264.58
Received: AAPL | Date: 2026-02-23 | Price: 266.18
Received: AAPL | Date: 2026-02-24 | Price: 272.14
Received: AAPL | Date: 2026-02-25 | Price: 274.23
Received: AAPL | Date: 2026-02-26 | Price: 272.95
Received: AAPL | Date: 2026-02-27 | Price: 264.18
Received: AAPL | Date: 2026-03-02 | Price: 264.72
Received: AAPL | Date: 2026-03-03 | Price: 263.75
Received: AAPL | Date: 2026-03-04 | Price: 262.52
Received: AAPL | Date: 2